In [1]:
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry, ModelType
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from utils.graph_builder import LocalizationGraphBuilder, traverse_namespaces
from adaptation.misc import NameAnonymizer
from schemas.similarity import SearchMethod
import os


In [2]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} spacy={'es': 'es_core_news_sm'} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/vectors.bin'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/bilstm_mean_cosine'} models=[<SearchMethod.SBERT: 'sbert'>, <SearchMethod.LSTM: 'lstm'>] allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000 faiss_data_dir='./faiss_data' adaptation_data_dir='./adaptation/data' localization_dir='./adaptation/localization'


In [3]:
adaptation_dir = "./adaptation"

localization_dir = os.path.join(adaptation_dir, "localization")
language_dir = os.path.join(localization_dir, "dialogue", "active")
structure_dir = os.path.join(localization_dir,  "structure", "modified")

database_dir = "./faiss_data"

data_dir = os.path.join(adaptation_dir, "data")
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [4]:
namespaces = traverse_namespaces(language_dir, languages)
print(namespaces)


['scene6/routeB/scene6LunchRouteB', 'names', 'scene1/scene1Bedroom2', 'generalDialogs', 'menus/titleScene', 'scene6/routeA/scene6PortalRouteA', 'scene6/routeB/scene6PoliceStationRouteB', 'menus/loginScene', 'scene6/routeB/scene6EndingRouteB', 'scene7/scene7Bedroom', 'scene3/scene3Break', 'scene6/routeA/scene6LunchRouteA', 'scene2/scene2Break', 'scene6/routeA/scene6EndingRouteA', 'transitions', 'scene4/scene4Backyard', 'scene6/routeA/scene6BedroomRouteA1', 'scene4/scene4Bedroom', 'dialogManager', 'scene1/scene1Bedroom1', 'scene3/scene3Bedroom', 'computer/loginScreen', 'scene6/scene6Bedroom', 'deviceInfo', 'menus/creditsScene', 'scene1/scene1Lunch1', 'scene6/routeB/scene6BedroomRouteB', 'scene2/scene2Bedroom', 'scene4/scene4Garage', 'scene6/scene6Livingroom', 'scene4/scene4Frontyard', 'computer/socialMediaScreen', 'computer/usernames', 'scene6/routeA/scene6BedroomRouteA2', 'scene1/scene1Break', 'scene5/scene5Bedroom', 'scene5/scene5Livingroom', 'scene1/scene1Lunch2', 'computer/captions',

In [5]:
backend = Backend(
    name_mapping=lambda lng, ns: os.path.join(
        language_dir,
        lng,
        f"{ns}.json"
    )
)

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [6]:
name_anonymizer = NameAnonymizer(
    names_path=spanish_names_path,
    whitelist_path=name_whitelist_path,
    replacement="[UNK]"
)


In [7]:
model_registry = ModelRegistry(languages)
model_registry.build_transformer(ModelType.SBERT)
model_registry.build_lstm()
model_registry.build_spacy()
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)


2026-07-13 06:01:01.267 | DEBUG    | services.model_registry:_create_loader:59 - Registering sbert loader for 'es'.
2026-07-13 06:01:01.268 | DEBUG    | services.model_registry:_create_loader:59 - Registering lstm loader for 'es'.
2026-07-13 06:01:01.268 | DEBUG    | services.model_registry:_create_loader:59 - Registering lstm calibrator loader for 'es'.
2026-07-13 06:01:01.269 | DEBUG    | services.model_registry:_create_loader:59 - Registering spaCy loader for 'es'.
2026-07-13 06:01:01.269 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-07-13 06:01:03.753 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'
2026-07-13 06:01:03.753 | DEBUG    | services.lazy_loader:model:16 - Loading lstm for 'es'...


Using device: cuda


2026-07-13 06:01:04.871 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded lstm for 'es'
2026-07-13 06:01:04.872 | DEBUG    | services.lazy_loader:model:16 - Loading spaCy for 'es'...
2026-07-13 06:01:05.158 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded spaCy for 'es'
2026-07-13 06:01:05.158 | DEBUG    | services.lazy_loader:model:16 - Loading lstm calibrator for 'es'...
2026-07-13 06:01:05.158 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded lstm calibrator for 'es'


In [8]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir=structure_dir,
    model_types=[
        SearchMethod.TFIDF,
        # SearchMethod.LSTM,
	]
)

builder.run()


2026-07-13 06:01:05.183 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 33 vectors
2026-07-13 06:01:05.194 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 40 vectors
2026-07-13 06:01:05.201 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 31 vectors
2026-07-13 06:01:05.210 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 48 vectors
2026-07-13 06:01:05.211 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 40 vectors
2026-07-13 06:01:05.226 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 35 vectors
2026-07-13 06:01:05.226 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 36 vectors
2026-07-13 06:01:05.243 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 39 vectors
2026-07-13 06:01:05.247 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 42 vectors
2026-07-13 06:01:05.261 | DEBUG    | controllers.retrievers.faiss:_fit:96 - Indexed 42 vectors
2026-07-13 06:01:05.264 | DEBUG    | controllers.r

Total visited nodes: 732


In [9]:
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
test_engine = multilingual.get_node_engine("es", SearchMethod.TFIDF)

print(test_engine.retrievers)

test_engine.load_all()

# test_engine.load_node("scene1Bedroom1_computer1_choices_similarity")

print(test_engine.retrievers)


2026-07-13 06:01:05.310 | DEBUG    | services.node_engine:load_node:67 - Loading FAISS node | method=tfidf | language=es | node=scene1Bedroom1_computer1_choices_similarity
2026-07-13 06:01:05.315 | SUCCESS  | services.node_engine:load_node:84 - Loaded node successfully.
2026-07-13 06:01:05.315 | DEBUG    | services.node_engine:load_node:67 - Loading FAISS node | method=tfidf | language=es | node=scene1Bedroom1_computer2_root
2026-07-13 06:01:05.319 | SUCCESS  | services.node_engine:load_node:84 - Loaded node successfully.
2026-07-13 06:01:05.319 | DEBUG    | services.node_engine:load_node:67 - Loading FAISS node | method=tfidf | language=es | node=scene1Bedroom2_computer_choices2_similarity
2026-07-13 06:01:05.322 | SUCCESS  | services.node_engine:load_node:84 - Loaded node successfully.
2026-07-13 06:01:05.322 | DEBUG    | services.node_engine:load_node:67 - Loading FAISS node | method=tfidf | language=es | node=scene1Classroom_part2_thanks_similarity
2026-07-13 06:01:05.326 | SUCCESS

{}
{'scene1Bedroom1_computer1_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001F700BE3BC0>, 'scene1Bedroom1_computer2_root': <controllers.retrievers.faiss.FaissRetriever object at 0x000001F727C78110>, 'scene1Bedroom2_computer_choices2_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001F727C79490>, 'scene1Classroom_part2_thanks_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001F727C79820>, 'scene2Break_part2_choice_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001F727C5B3B0>, 'scene3Bedroom_main_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001F727C59820>, 'scene4Backyard_mainConversation_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001F727C2B710>, 'scene4Bedroom_phone_root': <controllers.retrievers.faiss.FaissRetriever object at 0x000001F727C7AF60>, 'scene4Garage_phone1_root': <controllers.retrievers.fais

In [10]:
retriever = test_engine.get_retriever("scene1Bedroom1_computer1_choices_similarity")

retriever.search("Hola", 3)


127
127


(array([20, 14,  3], dtype=int32),
 array([1., 1., 0.], dtype=float32),
 array(['Hola, cual?', 'Hola, si, ¿Cómo lo sabes?', 'Ah djasdjashda guay'],
       dtype=object))